# [Kaggle] PatchCore Training - All Categories

Builds PatchCore memory banks for MVTec AD using a Kaggle GPU.
Uses pretrained ResNet features (no gradient optimization required).

### Kaggle Setup Instructions:
1. Requires **Internet: ON** (to download ResNet weights!).
2. Requires **GPU: ON** (feature extraction is much faster on GPU).
3. Add the **MVTec AD** dataset.
4. Upload your project source code (`src/` folder).

In [ ]:
# ---------------------------------------------------------
# Kaggle Environment Setup
# ---------------------------------------------------------
import os
import sys

KAGGLE_INPUT = '/kaggle/input'
KAGGLE_WORKING = '/kaggle/working'

# Update these paths based on your actual Kaggle Dataset names
MVTEC_PATH = '/kaggle/input/mvtec-ad'
SRC_PREFIX = '/kaggle/input/thesis-project-src'

if os.path.exists(SRC_PREFIX):
    sys.path.insert(0, SRC_PREFIX)
    print(f"Source code loaded from {SRC_PREFIX}")
else:
    print(f"WARNING: Source code folder not found at {SRC_PREFIX}.")


In [ ]:
# ---------------------------------------------------------
# Imports & Device Configuration
# ---------------------------------------------------------
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import time
from pathlib import Path
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
if DEVICE == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}")

MODELS_DIR = Path(KAGGLE_WORKING) / 'models'
FIGURES_DIR = Path(KAGGLE_WORKING) / 'figures'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Environment Patching

In [ ]:
try:
    from src import config as proj_cfg
    proj_cfg.DATA_DIR = Path(MVTEC_PATH).parent
    proj_cfg.MVTEC_DIR = Path(MVTEC_PATH)
    
    from src.data import create_mvtec_dataloaders
    from src.models.patchcore import create_patchcore
    
    MVTEC_CATEGORIES = [
        'bottle', 'cable', 'capsule', 'carpet', 'grid', 
        'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 
        'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
    ]
    print("Modules imported and config patched successfully.")
except ImportError as e:
    print(f"Import Error: {e}")

## Configuration

In [ ]:
CONFIG = {
    'batch_size': 16,        # Increased for GPU
    'backbone': 'resnet18',  # Must have internet ON to download!
    'k': 3,
    'subsample_ratio': 0.1,  # 10% subset (tradeoff between speed and accuracy)
}

CATEGORIES_TO_TRAIN = MVTEC_CATEGORIES
print(f"Training {len(CATEGORIES_TO_TRAIN)} categories.")

## Build Memory Bank Function

In [ ]:
def build_patchcore_category(category):
    print(f"\n{'='*60}")
    print(f"PatchCore: {category.upper()}")
    print(f"{'='*60}")

    try:
        train_loader, test_loader = create_mvtec_dataloaders(
            category, batch_size=CONFIG['batch_size'], return_mask=True
        )
    except Exception as e:
        print(f"Skipping {category}: {e}")
        return None

    model = create_patchcore(
        backbone=CONFIG['backbone'],
        k=CONFIG['k'],
        subsample_ratio=CONFIG['subsample_ratio']
    )

    start_time = time.time()
    print("  Extracting features and building memory bank...")
    model.fit(train_loader, device=DEVICE)
    fit_time = time.time() - start_time
    print(f"  Built in {fit_time:.1f}s | Size: {model.memory_bank.shape[0]}")

    print("  Evaluating on test set (k-NN search)...")
    model.eval()
    all_scores, all_labels = [], []
    
    # Note: k-NN inference can be memory intensive on Kaggle GPU if not done in batches.
    # The model's get_anomaly_score handles it per-batch, which is safe.
    with torch.no_grad():
        for img, mask, label in test_loader:
            img = img.to(DEVICE)
            scores = model.get_anomaly_score(img)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(label.numpy())

    try:
        auc = roc_auc_score(all_labels, all_scores)
        print(f"  >>> {category.upper()} ROC-AUC: {auc:.4f} <<<")
    except:
        auc = 0.0

    save_path = MODELS_DIR / f'patchcore_{category}_memory.pth'
    model.save_memory_bank(str(save_path))
    print(f"  Saved memory bank.")

    return {
        'category': category,
        'auc': auc,
        'fit_time_s': round(fit_time, 1),
        'memory_bank_size': model.memory_bank.shape[0]
    }

## Execute Run

In [ ]:
results = []
total_start = time.time()

for cat in CATEGORIES_TO_TRAIN:
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        
    res = build_patchcore_category(cat)
    if res:
        results.append(res)

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"ALL CATEGORIES COMPLETE in {total_elapsed/60:.1f} minutes")
print(f"{'='*60}")

## Summary Report

In [ ]:
if results:
    df = pd.DataFrame(results)
    csv_path = Path(KAGGLE_WORKING) / 'patchcore_results_all.csv'
    df.to_csv(csv_path, index=False)
    
    print("\n" + "="*40)
    print("PATCHCORE RESULTS SUMMARY")
    print("="*40)
    print(df.sort_values('auc', ascending=False).to_string(index=False))
    print(f"\nMean AUC: {df['auc'].mean():.4f}")
else:
    print("No successful runs.")